In [8]:
#!/usr/bin/env python3
"""
Construct an Industry-by-Industry (I×I) A-matrix from BEA Supply-Use tables.

Inputs:
  - Use_SUT_Detail.xlsx   (sheet "2017")
  - Supply_Detail.xlsx    (sheet "2017")

Outputs:
  - A_ixi_2017.csv
  - (optional) A_ixi_2017.xlsx

Method (Industry Technology Assumption, ITA):
  B = U * inv(diag(x))     where x = industry output (column sums of V)
  D = V.T * inv(diag(q))   where q = commodity output (row sums of V)
  A = D @ B
"""

from __future__ import annotations

import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd


SHEET = "2017"

# In BEA SUT detail files:
# row 5 contains column codes (industry codes / final demand codes)
# rows 6.. contain data, and col 0 is row code (commodity code)
HEADER_ROW = 5
DATA_START_ROW = 6


def _as_str(x) -> str:
    return "" if pd.isna(x) else str(x).strip()


def _is_industry_code(code: str) -> bool:
    """
    Heuristic filter:
      - exclude empty
      - exclude final demand codes that start with 'F'
      - exclude totals/controls that start with 'T'
    Keep everything else (BEA industry codes include digits and sometimes letters).
    """
    code = code.strip()
    if not code:
        return False
    if code.startswith("F"):
        return False
    if code.startswith("T"):
        return False
    if code.lower() in {"code", "commodity description"}:
        return False
    return True


def _read_table_matrix(
    xlsx_path: Path,
    sheet: str,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    """
    Reads a BEA-style table where:
      - header row (HEADER_ROW) has column codes
      - column 0 has row codes (commodities)
      - column 1 is descriptions (ignored)
      - numeric data start at DATA_START_ROW

    Returns:
      data_df: (commodities x columns) numeric DataFrame (still includes non-industry cols if present)
      row_codes: commodity codes in the same order as data_df.index
      col_codes: column codes in the same order as data_df.columns
    """
    raw = pd.read_excel(xlsx_path, sheet_name=sheet, header=None)

    # Column codes from header row
    col_codes = [_as_str(c) for c in raw.iloc[HEADER_ROW, :].tolist()]

    # Row codes from column 0
    row_codes = raw.iloc[DATA_START_ROW:, 0].map(_as_str).tolist()

    # Identify the last commodity row:
    # In BEA detail files, commodities end right before the first row code starting with 'T' or 'V' (e.g., T005, V00100).
    row_code_series = pd.Series(row_codes)
    end_idx = len(row_codes)  # exclusive
    for i, rc in enumerate(row_codes):
        if rc.startswith("T") or rc.startswith("V"):
            end_idx = i
            break

    row_codes = row_codes[:end_idx]

    # Extract numeric block: rows DATA_START_ROW .. DATA_START_ROW+end_idx-1
    # and all columns. We'll filter columns later.
    numeric_block = raw.iloc[DATA_START_ROW : DATA_START_ROW + end_idx, :].copy()

    # Drop the first two columns (row code + description), keep the rest as potential columns
    data = numeric_block.iloc[:, 2:].copy()
    data.columns = col_codes[2:]
    data.index = row_codes

    # Coerce to numeric, fill NaN with 0
    data = data.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    return data, row_codes, list(data.columns)


def construct_A_ixi(
    use_xlsx: Path,
    supply_xlsx: Path,
    sheet: str = SHEET,
) -> pd.DataFrame:
    """
    Constructs A (industry x industry) using ITA:
      - U from use table (commodities x industries)
      - V from supply table (commodities x industries)
      - x = column sums of V
      - q = row sums of V
      - B = U * inv(diag(x))
      - D = V.T * inv(diag(q))
      - A = D @ B
    """
    U_all, _, U_cols = _read_table_matrix(use_xlsx, sheet)
    V_all, _, V_cols = _read_table_matrix(supply_xlsx, sheet)

    # Filter to industry columns using intersection of codes present in both
    U_ind = [c for c in U_cols if _is_industry_code(c)]
    V_ind = [c for c in V_cols if _is_industry_code(c)]
    ind_codes = sorted(set(U_ind).intersection(V_ind))

    if not ind_codes:
        raise ValueError(
            "No overlapping industry columns found between Use and Supply files after filtering."
        )

    # Subset U and V to common industries
    U = U_all.loc[:, ind_codes].copy()  # commodities x industries
    V = V_all.loc[:, ind_codes].copy()  # commodities x industries

    # Ensure commodities align (intersection, then align)
    common_comm = U.index.intersection(V.index)
    if len(common_comm) == 0:
        raise ValueError("No overlapping commodity row codes found between Use and Supply tables.")

    U = U.loc[common_comm, :]
    V = V.loc[common_comm, :]

    # Outputs
    x = V.sum(axis=0)  # industry outputs (by column)
    q = V.sum(axis=1)  # commodity outputs (by row)

    # Guard against divide-by-zero industries/commodities
    # (drop zero-output industries; drop zero-output commodities)
    nonzero_ind = x[x != 0].index
    nonzero_com = q[q != 0].index

    U = U.loc[nonzero_com, nonzero_ind]
    V = V.loc[nonzero_com, nonzero_ind]
    x = x.loc[nonzero_ind]
    q = q.loc[nonzero_com]

    # B = U * inv(diag(x))  => divide each column j by x_j
    B = U.divide(x, axis=1)

    # D = V.T * inv(diag(q)) => divide each commodity row in V by q, then transpose
    D = V.divide(q, axis=0).T  # industries x commodities

    # A = D @ B  => industries x industries
    A = pd.DataFrame(D.to_numpy() @ B.to_numpy(), index=D.index, columns=B.columns)

    return A, U, V, x, q

In [12]:

use_xlsx = Path("../rampr/data/io/Use_SUT_Detail.xlsx")
supply_xlsx = Path("../rampr/data/io/Supply_Detail.xlsx")

A, U, V, x, q = construct_A_ixi(use_xlsx, supply_xlsx, sheet=SHEET)

out_csv = Path(f"A_ixi_{SHEET}.csv")
A.to_csv(out_csv, float_format="%.10g")
print(f"Wrote: {out_csv.resolve()}  shape={A.shape}")

# Optional Excel output
out_xlsx = Path(f"A_ixi_{SHEET}.xlsx")
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    A.to_excel(writer, sheet_name="A_ixi")
print(f"Wrote: {out_xlsx.resolve()}  shape={A.shape}")

# Quick numerical sanity checks
col_sums = A.sum(axis=0)
print(f"Column-sum range: min={col_sums.min():.4f}, max={col_sums.max():.4f}")

# Spectral radius check (can be expensive for 400x400 but OK)
eigvals = np.linalg.eigvals(A.to_numpy())
rho = float(np.max(np.abs(eigvals)))
print(f"Spectral radius rho(A) = {rho:.6f}  (must be < 1 for Leontief inverse to exist)")

L = np.linalg.inv(np.eye(401) - A)


Wrote: /Users/srey/projects/dev-rampr/rampr/scripts/A_ixi_2017.csv  shape=(401, 401)
Wrote: /Users/srey/projects/dev-rampr/rampr/scripts/A_ixi_2017.xlsx  shape=(401, 401)
Column-sum range: min=0.0000, max=1.5948
Spectral radius rho(A) = 0.533524  (must be < 1 for Leontief inverse to exist)


In [13]:
som = L.sum(axis=0)

In [16]:
np.median(som), som.max()

(np.float64(2.079050584524998), np.float64(4.7446049500256))

In [10]:
x

1111A0     38217.0
1111B0     67412.0
111200     18806.0
111300     29958.0
111400     19868.0
            ...   
S00201     16928.0
S00202     63412.0
S00203    292844.0
S00500    624289.0
S00600    405353.0
Length: 401, dtype: float64

In [11]:
q

1111A0     37922.0
1111B0     66790.0
111200     18582.0
111300     29731.0
111400     19868.0
            ...   
GSLGH      68698.0
GSLGO     836043.0
S00203    107059.0
S00401     10761.0
S00900      3468.0
Length: 399, dtype: float64

In [7]:
A.columns.values

array(['1111A0', '1111B0', '111200', '111300', '111400', '111900',
       '112120', '1121A0', '112300', '112A00', '113000', '114000',
       '115000', '211000', '212100', '212230', '2122A0', '212310',
       '2123A0', '213111', '21311A', '221100', '221200', '221300',
       '230301', '230302', '233210', '233230', '233240', '233262',
       '2332A0', '2332C0', '2332D0', '233411', '233412', '2334A0',
       '311111', '311119', '311210', '311221', '311224', '311225',
       '311230', '311300', '311410', '311420', '311513', '311514',
       '31151A', '311520', '311615', '31161A', '311700', '311810',
       '3118A0', '311910', '311920', '311930', '311940', '311990',
       '312110', '312120', '312130', '312140', '312200', '313100',
       '313200', '313300', '314110', '314120', '314900', '315000',
       '316000', '321100', '321200', '321910', '3219A0', '322110',
       '322120', '322130', '322210', '322220', '322230', '322291',
       '322299', '323110', '323120', '324110', '324121', '3241